# Sysdiagnose Analysis Framework — Jupyter Quickstart

This notebook demonstrates how to use sysdiagnose interactively in Jupyter.

## Setup
```bash
pip install sysdiagnose[jupyter]
```

In [ ]:
# Load the sysdiagnose Jupyter extension
%load_ext sysdiagnose.jupyter

## Case Management

In [ ]:
# List all available cases
%sd cases

In [ ]:
# Set the active case (replace with your case ID)
%sd use 1

In [ ]:
# Show case info
%sd info

## Parsing

In [ ]:
# List available parsers
%sd parsers

In [ ]:
# Run a parser — result is automatically loaded as a DataFrame
%sd parse ps

In [ ]:
# The result is available as _sd_result and sd_ps
sd_ps.head()

## Direct Python API

For more control, use the Python API directly.

In [ ]:
from sysdiagnose.jupyter.display import result_to_df
from sysdiagnose.jupyter import get_sd

sd = get_sd()
case_id = '1'  # adjust to your case

# Load any parsed result as a DataFrame
parsed_folder = sd.config.get_case_parsed_data_folder(case_id)
df = result_to_df(parsed_folder, 'shutdownlogs')
df

## Visualization

In [ ]:
from sysdiagnose.jupyter.viz import plot_timeline, plot_event_density, plot_time_gaps

# Load logarchive events (run %sd parse logarchive first)
df_log = result_to_df(parsed_folder, 'logarchive')

# Timeline scatter plot colored by module
plot_timeline(df_log, title='Logarchive Timeline')

In [ ]:
# Event density histogram
plot_event_density(df_log, freq='1h', title='Events per hour')

In [ ]:
# Detect time gaps (useful for identifying suspicious periods)
gaps = plot_time_gaps(df_log, min_gap_minutes=30)
gaps

In [ ]:
# WiFi geolocation map (run %sd analyse wifi_geolocation first)
from sysdiagnose.jupyter.viz import plot_wifi_map

df_wifi = result_to_df(parsed_folder, 'wifi_geolocation')
if df_wifi is not None:
    plot_wifi_map(df_wifi)

## Filtering and Correlation

Since results are DataFrames, you can use the full pandas ecosystem.

In [ ]:
# Example: find processes running from suspicious paths
df_ps = result_to_df(parsed_folder, 'ps')
if df_ps is not None:
    suspicious_paths = ['/private/var/db/com.apple.xpc.roleaccountd.staging']
    mask = df_ps.apply(lambda row: any(p in str(row) for p in suspicious_paths), axis=1)
    df_ps[mask]